In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sports-consolidated-dp./{data_source}/*.csv'


In [ ]:
# ---> BRONZE <---

df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)



In [ ]:
df.printSchema()
display(df.limit(10))

In [ ]:
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    ,mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_souce}')

In [ ]:
# ---> SILVER <---
